# 🧹 Module 3 — Class 1 CLASSWORK: Customer Database Rescue

**Lecture:** [https://bepro-aiml.github.io/aiml-platform/#/module/3/class/1](https://bepro-aiml.github.io/aiml-platform/#/module/3/class/1)

## 👋 Read this first — what is this notebook?

A **notebook** is a document that mixes:
- 📝 **Markdown cells** — text like this. They explain ideas.
- 💻 **Code cells** — Python you can run. The output shows up below.

To run a code cell: click on it, then press **Shift + Enter**.

## 🎯 Today's mission

It's your first day as a junior data analyst. Your manager drops a customer file on your desk and says:

> *"This was scraped together from three different sources over the last year. Clean it up before the marketing team uses it."*

The file is a mess: missing values, duplicates, inconsistent text, broken data types, impossible numbers. **Your job is to fix it.**

## 🧠 The Golden Rule

> **Never delete data silently.** Log what you dropped and why.

Bad data + a strong model = confidently wrong predictions. That is worse than no model at all. Data scientists spend ~80% of their time cleaning data — so do it well.

## 📦 Read every comment

Every code cell has comments — lines starting with `#`. **Comments are notes for humans. Python skips them.** They explain what each line does. **Don't skip them — they are the most important part of this notebook.**

## 🔧 Don't just READ — TRY

After each section there is a **🔧 Try It Yourself** block. Modify the code, re-run it, write down what changed. **You only learn this by breaking things.**

Let's go. 🚀

---
## 1. 🛠️ Generate the Dirty Dataset

We'll build a deliberately messy customer dataset in the notebook — no download needed. It contains every problem you'll see in real life.

Run the cell below. Then look at the output and try to spot at least 3 things that look wrong. **You'll fix them all today.**

In [ ]:
# ────────────────────────────────────────────────────────────
# STEP 1: Import the libraries we need.
# 'import' = bring in code that someone else already wrote.
# We give each library a short nickname using 'as'.
# ────────────────────────────────────────────────────────────
import pandas as pd      # pandas → tables (DataFrames). Nickname: pd
import numpy as np       # numpy  → numerical helpers. Nickname: np

# ────────────────────────────────────────────────────────────
# STEP 2: Set a random seed so everyone gets the SAME dirty data.
# Without a seed, each run produces different randomness.
# ────────────────────────────────────────────────────────────
rng = np.random.default_rng(42)

# ────────────────────────────────────────────────────────────
# STEP 3: Build a clean base of 60 customers, then SPOIL it.
# This is just stage prep — focus on the FINAL table, not the
# generation code.
# ────────────────────────────────────────────────────────────
n = 60
df = pd.DataFrame({
    'customer_id':  range(1001, 1001 + n),
    'name':         rng.choice(['Ali', 'Zara', 'Jasur', 'Madina', 'Otabek', 'Liam', 'Mia', 'Aisha'], n),
    'age':          rng.integers(18, 70, n).astype(float),
    'city':         rng.choice(['Tashkent', 'Samarkand', 'Bukhara'], n),
    'monthly_spend': rng.normal(450, 150, n).round(2),
    'signup_year':  rng.integers(2018, 2026, n),
    'plan':         rng.choice(['basic', 'pro', 'premium'], n),
    'referral_code': rng.choice(['REF01', 'REF02', None], n, p=[0.05, 0.05, 0.9])  # mostly missing
})

# ─── SPOIL #1: drop random ages and monthly_spend values
df.loc[rng.choice(n, 8, replace=False), 'age'] = np.nan
df.loc[rng.choice(n, 5, replace=False), 'monthly_spend'] = np.nan

# ─── SPOIL #2: store monthly_spend as STRING (a real-life nightmare)
df['monthly_spend'] = df['monthly_spend'].astype(str)
df.loc[2, 'monthly_spend'] = ' '          # blank string, not a real NaN
df.loc[7, 'monthly_spend'] = ''           # empty string

# ─── SPOIL #3: inconsistent capitalisation and trailing spaces
df.loc[1, 'city']  = 'tashkent'
df.loc[3, 'city']  = ' Tashkent '
df.loc[5, 'city']  = 'TASHKENT'
df.loc[10, 'plan'] = 'PRO'
df.loc[12, 'plan'] = 'Pro '

# ─── SPOIL #4: outliers and impossible values
df.loc[20, 'age'] = 999      # data entry error
df.loc[21, 'age'] = -5       # negative age, impossible
df.loc[22, 'monthly_spend'] = '99999.00'  # mega outlier — could be VIP or typo

# ─── SPOIL #5: exact duplicate rows (same customer logged twice)
df = pd.concat([df, df.iloc[[0, 4, 9]]], ignore_index=True)

# Save to disk so we can reload after cleaning if needed.
df.to_csv('customers_raw.csv', index=False)

# Show a peek so we can SEE the mess.
df.head(15)

### ✅ Check Your Understanding

Look at the table above. **Without running any more code**, answer:

1. List **three** things that look wrong with this data. (Hint: check the city column carefully.)
2. The `monthly_spend` column has a `.00` after the numbers. Does that mean it's a number (`float64`) or text (`object`)? **Trick question** — you can't tell yet. We'll prove it next.
3. Why does generating dirty data on purpose help us *learn*?

**Your answers:**

---
## 2. 🔍 Audit — Look Before You Touch

**Rule.** Before you change a single value, run an audit. You need to know what you're dealing with.

### 📚 The four audit commands you'll use forever

| Command | What it tells you |
| --- | --- |
| `df.shape` | `(rows, columns)` — size at a glance |
| `df.dtypes` | data type per column (`int64`, `float64`, `object`) |
| `df.info()` | dtypes **+** non-null counts in one block |
| `df.describe()` | summary stats for numeric columns |
| `df.isna().sum()` | count of missing values per column |

In [ ]:
# ────────────────────────────────────────────────────────────
# .shape returns a tuple: (rows, cols). No parentheses — it's an
# attribute, not a function.
# ────────────────────────────────────────────────────────────
print('Shape (rows, cols):', df.shape)

# .dtypes shows the data type of each column.
#   'object'  = text (string)
#   'int64'   = whole number
#   'float64' = decimal number
print('\nColumn dtypes:')
print(df.dtypes)

In [ ]:
# ────────────────────────────────────────────────────────────
# .info() combines shape + dtypes + non-null counts in one view.
# 'Non-Null Count' less than total rows = missing values exist.
# ────────────────────────────────────────────────────────────
df.info()

In [ ]:
# ────────────────────────────────────────────────────────────
# df.isna() returns a True/False table — True wherever a value
# is missing. .sum() then counts the Trues PER COLUMN.
# True = 1 and False = 0, so summing gives a count.
# ────────────────────────────────────────────────────────────
missing = df.isna().sum()
print('Missing per column:')
print(missing)

# Now express it as a fraction (more useful than raw count).
print('\nMissing fraction:')
print((df.isna().sum() / len(df)).round(2))

### ✅ Check Your Understanding

1. Look at the `dtypes` output. **`monthly_spend`** is `object`, not `float64`. Why? (Hint: we set it that way on purpose, to mimic real life.)
2. Which column has the **most missing values**? What fraction is missing?
3. **`describe()`** below — call it. Do the numbers in `age` look reasonable? Any value that screams *wrong*?

In [ ]:
# Run describe() — summary stats for numeric columns.
# Look at the 'min', 'max', and 'mean' for age. Anything off?
df.describe()

### 🔧 Try It Yourself — audit experiments

Below, run each of these one at a time. Write down what each tells you.

1. `df['city'].value_counts()` — count of each unique city. Look closely at the entries.
2. `df['plan'].value_counts()` — same for plan.
3. `df.duplicated().sum()` — how many exact duplicate rows exist?
4. `df.describe(include='object')` — summary for **text** columns. What is `unique` showing?

In [ ]:
# Your audit experiments — try each one
df['city'].value_counts()

**Observations from your audit:**

1. `city.value_counts()` reveals:
2. `plan.value_counts()` reveals:
3. Number of duplicate rows:
4. `describe(include='object')` shows:

---
## 3. 🔧 Fix Data Types First

**Why first?** If `monthly_spend` is text, you can't compute means, can't detect outliers, can't impute. Type-fix is the foundation.

### 📚 The conversion pattern

```python
df['col'] = pd.to_numeric(df['col'], errors='coerce')
```

- `pd.to_numeric` tries to turn each value into a number.
- `errors='coerce'` — when a value can't be converted (`' '`, `''`, `'abc'`), turn it into `NaN` instead of crashing.
- This is the **safest and most common** numeric conversion pattern in pandas.

In [ ]:
# ────────────────────────────────────────────────────────────
# Convert monthly_spend from text → number.
# Empty / blank strings become NaN automatically thanks to coerce.
# ────────────────────────────────────────────────────────────
df['monthly_spend'] = pd.to_numeric(df['monthly_spend'], errors='coerce')

# Confirm the type changed and check new missing count.
print('New dtype of monthly_spend:', df['monthly_spend'].dtype)
print('Missing in monthly_spend now:', df['monthly_spend'].isna().sum())

### ✅ Check Your Understanding

1. Before the conversion, `monthly_spend` was `object`. After, it's `float64`. Why does the missing count go **up** after conversion?
2. What would happen if we used `errors='raise'` (the default) instead of `'coerce'`? **Don't run it** — just predict.
3. Now that it's numeric, run `df['monthly_spend'].mean()`. Try the same line *before* the conversion (mentally) — would it have worked?

---
## 4. 🕳️ Handle Missing Values

From the lecture, in order of preference:

1. **Investigate the cause.** Missingness often carries signal.
2. **Add a `MissingIndicator`** column, then impute.
3. **Impute:** median for numeric, mode for categorical.
4. **Drop** only if >50% missing and unexplained.

### 📚 Why median, not mean?

**Median** is robust to outliers — half the values are below it, half above. **Mean** can be dragged sideways by one giant outlier (like our `99999.00` spender). For dirty data, prefer median.

### 📚 Why a missing-indicator?

Sometimes the fact that a value is missing is itself informative. A customer who didn't fill in `age` may behave differently from one who did. The model can learn that pattern — *if you give it the column*.

In [ ]:
# ────────────────────────────────────────────────────────────
# STEP 1: Add a missing-indicator BEFORE imputing.
# .isna() returns a True/False Series. We store it as a new column.
# ────────────────────────────────────────────────────────────
df['age_was_missing'] = df['age'].isna()

# ────────────────────────────────────────────────────────────
# STEP 2: Impute age with the median.
# .fillna(value) replaces every NaN with `value`.
# We compute the median FIRST so we can print it and remember.
# ────────────────────────────────────────────────────────────
age_median = df['age'].median()
print(f'Imputing age with median = {age_median}')
df['age'] = df['age'].fillna(age_median)

# ────────────────────────────────────────────────────────────
# STEP 3: Same routine for monthly_spend.
# ────────────────────────────────────────────────────────────
df['spend_was_missing'] = df['monthly_spend'].isna()
spend_median = df['monthly_spend'].median()
print(f'Imputing monthly_spend with median = {spend_median:.2f}')
df['monthly_spend'] = df['monthly_spend'].fillna(spend_median)

# ────────────────────────────────────────────────────────────
# STEP 4: Drop the column that's mostly missing — referral_code.
# In the audit it was ~90% None. Imputing 90% of values is just
# inventing data — drop it instead.
# ────────────────────────────────────────────────────────────
missing_pct = df['referral_code'].isna().mean()
print(f'\nreferral_code missing fraction = {missing_pct:.0%} → dropping')
df = df.drop(columns=['referral_code'])

# Confirm everything is now non-null
print('\nMissing per column AFTER cleanup:')
print(df.isna().sum())

### ✅ Check Your Understanding

1. Why did we add `age_was_missing` *before* `fillna`? What would go wrong if we did it after?
2. We dropped `referral_code` because it was ~90% missing. Why is imputing 90% of a column a bad idea?
3. **Categorical columns.** We had no missing categoricals this time. If `city` had been missing for 5 customers, what would you fill it with? (Hint: not mean.)

### 🔧 Try It Yourself — imputation experiments

Reload the raw data first (so we have something dirty to play with), then try:

1. Impute `age` with the **mean** instead of the median. Compare the imputed value to the median. Are they different? Why?
2. Impute `monthly_spend` with `0`. **Why is this almost always wrong?** (Hint: a customer who spent unknown amount ≠ a customer who spent zero.)
3. Try **forward-fill**: `df['age'] = df['age'].ffill()`. When would this make sense? When would it be wrong?

In [ ]:
# Reload the dirty version, then experiment.
raw = pd.read_csv('customers_raw.csv')
raw['monthly_spend'] = pd.to_numeric(raw['monthly_spend'], errors='coerce')

# Your experiment here
print('Mean age:  ', raw['age'].mean())
print('Median age:', raw['age'].median())

**Your observations:**

1. Mean vs median for age:
2. Why filling spend with 0 is wrong:
3. When `ffill` makes sense:

---
## 5. 🧽 Standardize Inconsistent Text

From the audit, our `city` column had `'Tashkent'`, `'tashkent'`, `' Tashkent '`, and `'TASHKENT'` — four spellings of the same place. The model would treat them as four different cities. We need exactly one.

### 📚 The two-step normalization

```python
df['city'] = df['city'].str.strip().str.lower()
```

- `.str.strip()` — remove leading/trailing whitespace.
- `.str.lower()` — lowercase everything.
- Apply both via the **`.str` accessor** — pandas' built-in string toolkit on Series.

These two steps fix 90% of dirty categoricals. For typos (`'Tasshkent'`) you'd need fuzzy matching — a topic for later.

In [ ]:
# ────────────────────────────────────────────────────────────
# Before — count the broken variants.
# ────────────────────────────────────────────────────────────
print('BEFORE:')
print(df['city'].value_counts())

# ────────────────────────────────────────────────────────────
# Strip whitespace, then lowercase, in one chained line.
# Apply to both city AND plan — same problem in both.
# ────────────────────────────────────────────────────────────
df['city'] = df['city'].str.strip().str.lower()
df['plan'] = df['plan'].str.strip().str.lower()

print('\nAFTER:')
print(df['city'].value_counts())
print()
print(df['plan'].value_counts())

### ✅ Check Your Understanding

1. How many distinct cities did `value_counts` show *before* normalization? How many *after*?
2. Why do we apply `strip()` **before** `lower()`? Would the order matter? (Try it both ways.)
3. **Stretch.** What about a typo like `'Tasshkent'`? Would these two steps catch it? Why or why not?

---
## 6. 👯 Remove Duplicates

An exact duplicate row is the same customer logged twice — counting them double would inflate every metric.

### 📚 Two ways to dedupe

```python
df.duplicated().sum()                    # how many duplicates exist?
df = df.drop_duplicates()                # drop exact duplicates
df = df.drop_duplicates(subset=['id'])   # drop dupes by KEY column
```

- **No `subset`** → entire row must match exactly.
- **With `subset=['col']`** → drops dupes based on that column only. Keeps the FIRST occurrence by default.

In [ ]:
# ────────────────────────────────────────────────────────────
# Count duplicates before dropping — log every change.
# ────────────────────────────────────────────────────────────
print('Total rows BEFORE:', len(df))
print('Duplicates by all cols   :', df.duplicated().sum())
print('Duplicates by customer_id:', df.duplicated(subset=['customer_id']).sum())

# customer_id is the natural key — same id = same customer.
df = df.drop_duplicates(subset=['customer_id']).reset_index(drop=True)

print('\nTotal rows AFTER :', len(df))
print('Duplicates remaining:', df.duplicated().sum())

### ✅ Check Your Understanding

1. We chose `subset=['customer_id']` instead of just `drop_duplicates()`. **Why is this safer for customer data?** (Hint: imagine the same customer logged in twice and one of the rows has a typo.)
2. By default `keep='first'` — the first occurrence is kept. What does `keep='last'` do? When would you use it?
3. What does `.reset_index(drop=True)` do, and why bother?

---
## 7. 🚨 Detect and Handle Outliers

We saw `age = 999` and `age = -5` earlier. Time to deal with them.

### 📚 The Tukey IQR rule

```
Q1 = 25th percentile
Q3 = 75th percentile
IQR = Q3 - Q1                       (interquartile range)
outlier if  value < Q1 - 1.5*IQR
          or value > Q3 + 1.5*IQR
```

This is the **Tukey rule** — a classical, robust definition of "unusually high or low."

### 📚 Two strategies (NEVER blindly delete)

1. **Replace impossible values with NaN, then re-impute.** Use this for clearly broken values (negative age, age=999). They are *errors*, not real data.
2. **Winsorize (cap)** real-but-extreme values to a max threshold. Keeps the customer in the dataset, prevents the value from skewing the mean.

🚨 **From the lecture:** *A fraud transaction IS an outlier — it's what you want to detect.* Outliers are not always errors. Think before you delete.

In [ ]:
# ────────────────────────────────────────────────────────────
# STEP 1: Impossible ages — these are errors, not outliers.
# Realistic human age range: 0 to ~120. Anything outside is a bug.
# Replace with NaN, then fill with the median we computed earlier.
# ────────────────────────────────────────────────────────────
impossible = (df['age'] < 0) | (df['age'] > 120)
print(f'Impossible ages found: {impossible.sum()}')
print(f'Values: {df.loc[impossible, "age"].tolist()}')

df.loc[impossible, 'age'] = np.nan
df['age'] = df['age'].fillna(df['age'].median())

In [ ]:
# ────────────────────────────────────────────────────────────
# STEP 2: Detect monthly_spend outliers via the IQR rule.
# ────────────────────────────────────────────────────────────
Q1 = df['monthly_spend'].quantile(0.25)
Q3 = df['monthly_spend'].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print(f'Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}')
print(f'Outlier band: [{lower:.2f}, {upper:.2f}]')

is_outlier = (df['monthly_spend'] < lower) | (df['monthly_spend'] > upper)
print(f'\nOutliers in monthly_spend: {is_outlier.sum()}')
print(df.loc[is_outlier, ['customer_id', 'monthly_spend']])

In [ ]:
# ────────────────────────────────────────────────────────────
# STEP 3: Winsorize (cap) instead of delete.
# A 99999 spender might be a real VIP — keeping them but capping
# their spend at the upper threshold prevents one row from
# dragging the mean.
#
# .clip(lower, upper) replaces values below `lower` with `lower`
# and values above `upper` with `upper`. Anything inside is untouched.
# ────────────────────────────────────────────────────────────
df['monthly_spend'] = df['monthly_spend'].clip(lower=lower, upper=upper)

print('After winsorizing:')
print(df['monthly_spend'].describe())

### ✅ Check Your Understanding

1. We treated `age = 999` as an **error** (replace with NaN), but `monthly_spend = 99999` as an **outlier** (winsorize). What's the difference, and why use different strategies?
2. Compare `df['monthly_spend'].describe()` *before* and *after* the clip. Which statistic changed the most? Why?
3. **Counter-example.** You're building a fraud detector. You see a transaction 50× the customer's average. Should you cap it? **Why or why not?**

### 🔧 Try It Yourself — outlier experiments

1. Try the **Z-score rule** instead of IQR. Compute `z = (x - mean) / std` and flag anything with `|z| > 3`. Do you get the same outliers as IQR?
2. Change `1.5 * IQR` to `3.0 * IQR`. How many outliers now? When would a stricter rule make sense?
3. **Drop instead of cap.** Try `df = df[~is_outlier]` (the `~` means NOT). How many rows did you lose? Is this safe for our use case?

In [ ]:
# Your outlier experiments here
# Hint for Z-score:
# z = (raw['monthly_spend'] - raw['monthly_spend'].mean()) / raw['monthly_spend'].std()
# z.abs() > 3


---
## 8. ✅ Final Audit + Save Cleaned Data

Re-run the audit. Compare it to the very first audit. **Every line should now make sense.**

In [ ]:
# ────────────────────────────────────────────────────────────
# Re-run the full audit on the cleaned data.
# Every column should have:
#   - the right dtype
#   - zero missing values
#   - sensible min/max
# ────────────────────────────────────────────────────────────
print('Shape:', df.shape)
print()
df.info()
print()
print('Missing per column:')
print(df.isna().sum())

In [ ]:
# Numeric summary should now show realistic ranges
df.describe()

In [ ]:
# ────────────────────────────────────────────────────────────
# Save to a NEW file. Never overwrite the raw input.
# index=False = don't write the row numbers as a column.
# ────────────────────────────────────────────────────────────
df.to_csv('customers_cleaned.csv', index=False)
print('Cleaned data saved to customers_cleaned.csv')
print(f'Final shape: {df.shape}')

### ✅ Final Check Your Understanding

1. The raw file had **63 rows**. The cleaned file has fewer. **How many rows did you drop, and why?**
2. The cleaned file has **more columns** than the raw — `age_was_missing` and `spend_was_missing`. **Why keep these?**
3. **The Golden Rule:** *never delete data silently.* List, in order, every change you made today. (This is the kind of log a senior engineer expects in a PR description.)

**Your change log (one line per step):**

1.
2.
3.
4.
5.
6.
7.

---
## 9. 🛠 Your Turn — Apply It to Real Data

**Take-home:** repeat this entire flow on the **Telco Customer Churn** dataset.

🔗 [https://www.kaggle.com/datasets/blastchar/telco-customer-churn](https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

It has the same problems as our toy dataset — plus a couple of bonus ones (the `TotalCharges` column is stored as a string with blank entries — sound familiar?). Apply the same 7 steps and produce `telco_cleaned.csv`.

Full assignment spec is in the lecture's *Activities & Assignments* section.

---
## 🏁 Mission Debrief

| Step | Pandas command | What it answers |
| --- | --- | --- |
| 🔍 Audit | `df.info()`, `df.isna().sum()`, `df.describe()` | *What am I looking at? What's broken?* |
| 🔧 Type fix | `pd.to_numeric(df['c'], errors='coerce')` | *Make text-numbers into real numbers* |
| 🕳️ Impute | `df['c'].fillna(df['c'].median())` | *Fill missing with a sensible default* |
| 🚩 Track | `df['c_was_missing'] = df['c'].isna()` | *Don't lose the missingness signal* |
| 🧽 Normalize | `df['c'].str.strip().str.lower()` | *One spelling per category* |
| 👯 Dedupe | `df.drop_duplicates(subset=['id'])` | *One row per customer* |
| 🚨 Outliers | `df['c'].clip(lower, upper)` | *Cap extremes without losing rows* |

### 🎓 Vocab Recap

- **NaN** — "Not a Number" — pandas' missing-value marker.
- **Imputation** — filling missing values with a guess (median, mode, model prediction).
- **MissingIndicator** — a binary column flagging "was this originally missing?"
- **IQR (Interquartile Range)** — `Q3 - Q1`. Foundation of Tukey's outlier rule.
- **Winsorize / clip** — cap extreme values to a threshold instead of deleting them.
- **`pd.to_numeric(..., errors='coerce')`** — safe text-to-number conversion that produces NaN on failure.
- **`.str` accessor** — pandas' string toolkit on Series (`.strip`, `.lower`, `.contains`, ...).
- **`drop_duplicates(subset=...)`** — dedupe by key, not full row.
- **MCAR / MAR / MNAR** — three types of missingness. Lecture has the full taxonomy.

### Submit
Save this notebook as `Module3_Class1_<YourName>.ipynb`. Submit a PR to your group repo at `module-3/class_1/submissions/<YourName>/`. Include `customers_cleaned.csv` if you saved it.

🧹 *See you in Class 2 — Encoding and Scaling.*